# Dense Retriever
> Retrieve context items by embedding similarity using a vector store.

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `DenseRetriever`, `InMemoryVectorStore`, `InMemoryContextStore`, `QueryBundle` |
| Difficulty | Beginner |

`DenseRetriever` embeds queries and documents into a shared vector space and
retrieves the closest matches from an `InMemoryVectorStore`.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.retrieval import DenseRetriever
from anchor.storage import InMemoryVectorStore, InMemoryContextStore
from anchor.models import ContextItem, SourceType, QueryBundle

## Create Stores and Embedding Function
We use in-memory stores and a simple hash-based embedding for demonstration.

In [ ]:
store = InMemoryVectorStore()
ctx_store = InMemoryContextStore()

def embed_fn(text: str) -> list[float]:
    """Deterministic pseudo-embedding for demonstration."""
    padded = text[:128].ljust(128)
    return [hash(c) % 100 / 100.0 for c in padded]

vec = embed_fn("hello world")
print(f"Embedding dimension: {len(vec)}")
print(f"First 5 values: {vec[:5]}")

## Initialize the Dense Retriever
Wire up the vector store, context store, and embedding function.

In [ ]:
retriever = DenseRetriever(
    vector_store=store,
    context_store=ctx_store,
    embed_fn=embed_fn,
)

print(f"Retriever type: {type(retriever).__name__}")

## Index Documents
Create `ContextItem` objects and index them into the retriever.

In [ ]:
topics = ["python", "rust", "go", "javascript", "typescript"]

items = [
    ContextItem(
        id=f"doc-{i}",
        content=f"Document about {topic}: {topic} is a popular programming language.",
        source=SourceType.RETRIEVAL,
        score=0.0,
        priority=5,
        token_count=10,
    )
    for i, topic in enumerate(topics)
]

retriever.index(items)

print(f"Indexed {len(items)} documents:")
for item in items:
    print(f"  {item.id}: {item.content[:50]}")

## Retrieve by Query
Wrap the query in a `QueryBundle` and call `.retrieve()`.

In [ ]:
query = QueryBundle(query_str="programming languages")
results = retriever.retrieve(query, top_k=3)

print(f"Query: '{query.query_str}'")
print(f"Top {len(results)} results:\n")
for i, item in enumerate(results):
    print(f"  [{i+1}] score={item.score:.4f}  {item.content[:60]}")

## Vary top_k
Adjust how many results are returned.

In [ ]:
for k in [1, 3, 5]:
    results = retriever.retrieve(query, top_k=k)
    print(f"top_k={k}: returned {len(results)} results")

## Key Takeaways
- `DenseRetriever` uses embedding similarity for semantic search
- Provide `vector_store`, `context_store`, and `embed_fn` at construction
- `.index(items)` adds `ContextItem` objects to the vector store
- `.retrieve(query, top_k=N)` returns the N closest matches
- Swap `InMemoryVectorStore` for a persistent backend in production

**Next:** [Sparse Retriever](02_sparse_retriever.ipynb)